# Dendros Quickstart

This notebook demonstrates the main workflows of the **`dendros`** library
for reading and exploring [Galacticus](https://github.com/galacticusorg/galacticus)
HDF5 output files.

We begin by creating a small synthetic HDF5 file that mimics the Galacticus
output structure, so you can run this notebook without needing a real
Galacticus run.

In [ ]:
import tempfile
from pathlib import Path

import h5py
import numpy as np

# -----------------------------------------------------------------------
# Create a minimal synthetic Galacticus-like HDF5 file
# -----------------------------------------------------------------------

tmpdir = tempfile.mkdtemp()
fpath = str(Path(tmpdir) / "galacticus.hdf5")

rng = np.random.default_rng(42)
N1, N2 = 500, 300  # galaxy counts for two snapshots

with h5py.File(fpath, "w") as f:
    f.attrs["statusCompletion"] = 0

    outputs = f.create_group("Outputs")

    # --- Output 1 (z=0) ---
    o1 = outputs.create_group("Output1")
    o1.attrs["outputTime"] = 13.8          # Gyr
    o1.attrs["outputExpansionFactor"] = 1.0
    nd1 = o1.create_group("nodeData")
    ds = nd1.create_dataset("basicMass", data=rng.lognormal(26.0, 1.0, N1))
    ds.attrs["comment"] = "Virial mass of the halo"
    ds.attrs["unitsInSI"] = 1.989e30
    ds = nd1.create_dataset("diskMassStellar", data=rng.lognormal(23.0, 1.0, N1))
    ds.attrs["comment"] = "Disk stellar mass"
    ds.attrs["unitsInSI"] = 1.989e30
    ds = nd1.create_dataset("diskRadius", data=rng.lognormal(-3.0, 0.5, N1))
    ds.attrs["comment"] = "Half-mass radius of the disk"
    ds.attrs["unitsInSI"] = 3.086e22

    # --- Output 2 (z=1) ---
    o2 = outputs.create_group("Output2")
    o2.attrs["outputTime"] = 5.9
    o2.attrs["outputExpansionFactor"] = 0.5
    nd2 = o2.create_group("nodeData")
    ds = nd2.create_dataset("basicMass", data=rng.lognormal(25.5, 1.0, N2))
    ds.attrs["comment"] = "Virial mass of the halo"
    ds.attrs["unitsInSI"] = 1.989e30
    ds = nd2.create_dataset("diskMassStellar", data=rng.lognormal(22.5, 1.0, N2))
    ds.attrs["comment"] = "Total stellar mass"
    ds.attrs["unitsInSI"] = 1.989e30
    ds = nd2.create_dataset("diskRadius", data=rng.lognormal(-3.2, 0.5, N2))
    ds.attrs["comment"] = "Half-mass radius of the disk"
    ds.attrs["unitsInSI"] = 3.086e22

print(f"Synthetic file written to: {fpath}")

## Opening a file

In [ ]:
import sys
import os
module_path = os.path.abspath('/carnegie/nobackup/users/abenson/Galacticus/dendros/src')
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from dendros import open_outputs

c = open_outputs(fpath)
print(c)

## Checking completion status

In [ ]:
c.validate_completion()  # raises RuntimeError if the run did not complete
print("Run completed successfully.")

## Listing outputs

`list_outputs()` returns an `astropy.table.Table` (default), a
`pandas.DataFrame` or a `tabulate` string with time, scale factor, and redshift for each snapshot.

In [ ]:
tbl = c.list_outputs()
print(tbl)

In [ ]:
# Access individual output metadata
for meta in c.outputs:
    print(f"{meta.name}: z = {meta.redshift:.3f}, t = {meta.time:.2f} Gyr")

## Listing properties

`list_properties()` shows the datasets available in the `nodeData` group,
together with their description and SI conversion factor.

In [ ]:
props = c.list_properties("Output1",format="tabulate")
print(props)

## Reading datasets

In [ ]:
# Read by dataset path — same string used as dict key
data = c.read("Output1", ["nodeData/basicMass", "nodeData/diskMassStellar"])
print("basicMass shape:", data["nodeData/basicMass"].shape)

# Or use a dict for custom labels
data = c.read(
    "Output1",
    {"Mhalo": "nodeData/basicMass", "Mstar": "nodeData/diskMassStellar"},
)
print("Mhalo[:5] =", data["Mhalo"][:5])

## Filtering galaxies

Pass a boolean mask or integer index array as `where`.

In [ ]:
all_data = c.read("Output1", {"Mhalo": "nodeData/basicMass"})
massive = all_data["Mhalo"] > 1.0e12

filtered = c.read(
    "Output1",
    {"Mhalo": "nodeData/basicMass", "Mstar": "nodeData/diskMassStellar"},
    where=massive,
)
print(f"Selected {massive.sum()} of {len(all_data['Mhalo'])} galaxies")

## h5py-like browsing

In [ ]:
print("Top-level keys:", c.keys())

grp = c["Outputs/Output1"]
print("Output1 attrs:", grp.attrs)
print("Output1 keys:", grp.keys())

ds = c["Outputs/Output1/nodeData/basicMass"]
print(f"haloMass dtype={ds.dtype}  shape={ds.shape}")

## Multi-file (MPI) example

Create two synthetic MPI-split files and open them together.

In [ ]:
mpi0 = str(Path(tmpdir) / "galacticus_MPI:0000.hdf5")
mpi1 = str(Path(tmpdir) / "galacticus_MPI:0001.hdf5")

for rank, path in enumerate([mpi0, mpi1]):
    with h5py.File(path, "w") as f:
        f.attrs["statusCompletion"] = 0
        grp = f.create_group("Outputs/Output1")
        grp.attrs["outputTime"] = 13.8
        grp.attrs["outputExpansionFactor"] = 1.0
        nd = grp.create_group("nodeData")
        start = rank * 250
        ds = nd.create_dataset("basicMass", data=rng.lognormal(26.0, 1.0, 250))
        ds.attrs["comment"] = "Virial mass of the halo"
        ds.attrs["unitsInSI"] = 1.989e30

# open_outputs auto-detects the peer file from any single rank
with open_outputs(mpi0) as mc:
    print("Files in collection:", mc.files)
    data = mc.read("Output1", ["nodeData/basicMass"])
    print("Total galaxies from both ranks:", len(data["nodeData/basicMass"]))

## Clean up

In [ ]:
c.close()
import shutil
shutil.rmtree(tmpdir)
print("Done.")